# Phase 4 — SAP-aware Template Retrieval
Reuse existing Phase 3 outputs, no API key or DeepSeek requests. Read-only approved Generic Library. Candidate rankings and gap analysis, not shell generation. Outputs include confidential evidence; share only if authorized.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'generic-retrieval-v2'
ROOT = Path('/content/tfl-generic-retrieval-v2')
if not ROOT.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)])
else:
    current = subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'], text=True).strip()
    assert current == BRANCH, f'Wrong branch: {current}'
    subprocess.check_call(['git','-C',str(ROOT),'pull','--ff-only'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[notebook,test]','sentence-transformers>=3,<6'])
SRC = (ROOT / 'src').resolve()
assert (SRC / 'tfl_agent/tools/sap_hybrid.py').is_file(), SRC
if str(SRC) in sys.path:
    sys.path.remove(str(SRC))
sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
cached = sys.modules.get('tfl_agent')
if cached is not None and not Path(cached.__file__).resolve().is_relative_to(SRC):
    raise RuntimeError('Wrong tfl_agent checkout already imported. Restart session and run all.')
import tfl_agent
from tfl_agent.agent.sap_scope_pilot import run_scope_retrieval
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC)
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent package loaded:', tfl_agent.__file__)
from tfl_agent.agent.phase4_pipeline import run_phase4

## 1. Read the saved Phase 3 output from Drive
The latest timestamped Phase 3 ZIP is used by default. Set ZIP_NAME to a specific filename if desired. Check SAP SHA-256 to prevent accidental reuse of another study.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile, json, hashlib
from datetime import datetime, timezone
DRIVE_SAP_RELATIVE_PATH = 'Colab Notebooks/agent/midterm demo/SAP_001_00.pdf'
SAP_PATH = Path('/content/drive/MyDrive') / DRIVE_SAP_RELATIVE_PATH
PHASE3_DIR = SAP_PATH.parent / 'Phase3_Outputs'
ZIP_NAME = '' #@param {type:"string"}
available = sorted(PHASE3_DIR.glob('*.zip'))
if not available:
    raise FileNotFoundError(f'No Phase 3 ZIP in {PHASE3_DIR}. Run Notebook 07 and save its output first.')
INPUT_ZIP = PHASE3_DIR / ZIP_NAME if ZIP_NAME else available[-1]
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SRC3 = Path('/content/Phase3_Input') / RUN_ID
SRC3.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(INPUT_ZIP) as z:
    for member in z.infolist():
        if not (SRC3/member.filename).resolve().is_relative_to(SRC3.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(SRC3)
s3 = json.loads((SRC3/'Summary.json').read_text())
assert hashlib.sha256(SAP_PATH.read_bytes()).hexdigest() == s3['sap_sha256'], 'Wrong SAP for Phase 3 ZIP'
OUT = Path('/content/SAP_Aware_Phase4') / RUN_ID
print('Input:', INPUT_ZIP)
print('SAP hash verified:', s3['sap_sha256'])

## 2. Run normalization, template/preset retrieval and gap analysis
All approved same-type templates are considered. Scores are provisional; ambiguity remains reviewable.

In [ ]:
import pandas as pd
from IPython.display import display
summary, results, plans = run_phase4(SRC3, ROOT/'generic_template_library', OUT)
print(json.dumps(summary, indent=2))
for result, plan in zip(results, plans):
    print(result['identity']['title'])
    print('Baseline:', result['baseline']['template_id'], result['baseline']['preset_id'])
    print('SAP-aware:', result['template_id'], result['preset_id'], result['flags'])
    display(pd.json_normalize(result['candidates'])[['template_id','preset_id','score','components.keyword','components.semantic','components.structure','components.statistics','components.preset','structural_conflict']])
    display(pd.DataFrame(plan['actions'])[['action','element','value','reason']] if plan['actions'] else pd.DataFrame())
    display(pd.DataFrame(plan['coverage_review']))

## 3. Run targeted regression tests

In [ ]:
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_sap_aware_retrieval_phase4.py','tests/test_scoped_specification_phase3.py','tests/test_sap_scope_phase21.py','tests/test_sap_hybrid_phase2.py','tests/test_generic_library_phase1.py'],cwd=ROOT)

## 4. Export results
Save a timestamped ZIP to Drive and download. No source artifacts or keys are committed to GitHub.

In [ ]:
import shutil
from google.colab import files
ZIP_PATH = Path(shutil.make_archive(str(OUT), 'zip', OUT))
destination = SAP_PATH.parent / 'Phase4_Outputs'
destination.mkdir(parents=True, exist_ok=True)
saved = Path(shutil.copy2(ZIP_PATH, destination / ZIP_PATH.name))
print('Saved to Drive:', saved)
files.download(str(ZIP_PATH))